In [109]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import warnings
warnings.filterwarnings('ignore')

In [110]:
plt.style.use('ggplot')

pd.set_option('display.max_columns', None)


In [111]:
articles = pd.read_csv('../Data/articles.csv')
transactions = pd.read_csv('../Data/transactions_train.csv',nrows=400000)

In [112]:
print("Articles:", articles.shape)
print("Transactions:", transactions.shape)

Articles: (105542, 25)
Transactions: (400000, 5)


In [113]:
transactions["t_dat"] = pd.to_datetime(
    transactions["t_dat"]
)

## Feature Engineering and Data Preprocessing

In [114]:
# Merging datasets

data = transactions.merge(
    articles,
    on="article_id",
    how="left"
)

print(data.shape)

(400000, 29)


In [115]:
# Filtering active customers

customer_counts = data["customer_id"].value_counts()

active_customers = customer_counts[
    customer_counts >= 5
].index

data = data[
    data["customer_id"].isin(active_customers)
]

print(data.shape)

(251302, 29)


In [116]:
# Filtering Popular products

product_counts = data["article_id"].value_counts()

popular_products = product_counts[
    product_counts >= 20
].index

data = data[
    data["article_id"].isin(popular_products)
]

print(data.shape)

(160276, 29)


In [117]:
# Creating Customer-Product Matrix

customer_product_matrix = pd.crosstab(
    data["customer_id"],
    data["article_id"]
)

print(customer_product_matrix.shape)

(27057, 3032)


In [118]:
# Calculating Product Similarity

from sklearn.metrics.pairwise import cosine_similarity

product_similarity = cosine_similarity(
    customer_product_matrix.T
)

In [119]:
## Creating Similarity DataFrame

similarity_df = pd.DataFrame(
    product_similarity,
    index=customer_product_matrix.columns,
    columns=customer_product_matrix.columns
)

similarity_df.head()

article_id  108775015  108775044  108775051  110065011  111565001  111586001  \
article_id                                                                     
108775015    1.000000   0.458223   0.010555   0.000000   0.000000   0.042658   
108775044    0.458223   1.000000   0.023080   0.000000   0.000000   0.000000   
108775051    0.010555   0.023080   1.000000   0.000000   0.000000   0.000000   
110065011    0.000000   0.000000   0.000000   1.000000   0.016473   0.000000   
111565001    0.000000   0.000000   0.000000   0.016473   1.000000   0.000000   

article_id  111593001  111609001  118458039  120129001  123173001  146730001  \
article_id                                                                     
108775015    0.006455        0.0        0.0        0.0        0.0        0.0   
108775044    0.000000        0.0        0.0        0.0        0.0        0.0   
108775051    0.000000        0.0        0.0        0.0        0.0        0.0   
110065011    0.000000        0.0        0.0        0.0        0.0        0.0   
111565001    0.006132        0.0        0.0        0.0        0.0        0.0   

article_id  148033001  156231001  158340001  160442007  160442010  160442043  \
article_id                                                                     
108775015    0.000000   0.014586   0.000000   0.027366        0.0   0.011994   
108775044    0.000000   0.000000   0.000000   0.003627        0.0   0.000000   
108775051    0.000000   0.000000   0.000000   0.000000        0.0   0.000000   
110065011    0.000000   0.000000   0.000000   0.000000        0.0   0.000000   
111565001    0.147508   0.008313   0.014014   0.000000        0.0   0.000000   

article_id  179123001  179208001  179393001  179393018  179950001  179950017  \
article_id                                                                     
108775015    0.015312        0.0   0.011259        0.0        0.0        0.0   
108775044    0.000000        0.0   0.008207        0.0        0.0        0.0   
108775051    0.000000        0.0   0.000000        0.0        0.0        0.0   
110065011    0.000000        0.0   0.000000        0.0        0.0        0.0   
111565001    0.000000        0.0   0.016042        0.0        0.0        0.0   

article_id  181448022  181448102  181448109  182909001  186262006  186264014  \
article_id                                                                     
108775015    0.006464        0.0        0.0        0.0        0.0        0.0   
108775044    0.000000        0.0        0.0        0.0        0.0        0.0   
108775051    0.000000        0.0        0.0        0.0        0.0        0.0   
110065011    0.000000        0.0        0.0        0.0        0.0        0.0   
111565001    0.000000        0.0        0.0        0.0        0.0        0.0   

article_id  188183001  188183009  188183012  189616001  189616006  189616007  \
article_id                                                                     
108775015         0.0        0.0        0.0        0.0   0.006445   0.000000   
108775044         0.0        0.0        0.0        0.0   0.000000   0.004808   
108775051         0.0        0.0        0.0        0.0   0.000000   0.000000   
110065011         0.0        0.0        0.0        0.0   0.000000   0.000000   
111565001         0.0        0.0        0.0        0.0   0.000000   0.000000   

article_id  189616008  189626001  189634001  189654001  189691033  189691044  \
article_id                                                                     
108775015         0.0        0.0   0.006472        0.0        0.0   0.000000   
108775044         0.0        0.0   0.000000        0.0        0.0   0.000000   
108775051         0.0        0.0   0.000000        0.0        0.0   0.016163   
110065011         0.0        0.0   0.000000        0.0        0.0   0.000000   
111565001         0.0        0.0   0.000000        0.0        0.0   0.000000   

article_id  189691067  189691075  190021003  194037001  198518010  198518

In [120]:
# Recommending Function

def recommend_products(article_id, top_n=10):

    if article_id not in similarity_df.columns:
        return []

    recommendations = (
        similarity_df[article_id]
        .sort_values(ascending=False)
        .iloc[1:top_n+1]
        .index
        .tolist()
    )

    return recommendations

In [121]:
# Testing the recommendation function

sample_product = data["article_id"].iloc[0]

recommendations = recommend_products(
    sample_product
)

recommendations

[685687004,
 685687001,
 685687002,
 654100005,
 537346026,
 630672001,
 582077002,
 562245001,
 622955001,
 673677003]

In [122]:
article_lookup = (
    articles
    .set_index("article_id")["prod_name"]
    .to_dict()
)

In [123]:
# Showing recommended product names

print("Selected Product:")
print(article_lookup[sample_product])

print("\nRecommended Products:\n")

for item in recommendations:
    print(article_lookup.get(item))

Selected Product:
W YODA KNIT OL OFFER

Recommended Products:

W YODA KNIT OL OFFER
W YODA KNIT OL OFFER
W YODA KNIT OL OFFER
Flirty Enya necklace RT
Ridge
SALLY BLOUSE
Jacket Oversized
Luna skinny RW
Carolina ankle
Henry polo (1)


In [124]:
# Saving The Model

import pickle

with open("../models/similarity.pkl", "wb") as f:
    pickle.dump(similarity_df, f)

In [125]:
# article metadata

with open("../models/articles.pkl", "wb") as f:
    pickle.dump(articles, f)

In [126]:
import os

os.listdir("../models")

['articles.pkl', 'similarity.pkl']